# Hiring

## Dataset

We use the ACSEmployment dataset from the Folktables benchmark, which is derived from the U.S. Census American Community Survey (ACS). The task is a binary classification problem where the goal is to predict whether a person is employed (1) or not employed (0) based on demographic and socioeconomic features such as age, education, marital status, and work-related attributes. The dataset also includes sensitive attributes — race and sex — which are used to evaluate fairness.

### Race (RAC1P)

The variable RAC1P encodes a person’s self-identified race in the U.S. Census ACS data.

- 1: White alone
- 2: Black or African American alone
- 3: American Indian or Alaska Native alone
- 4: Alaska Native alone
- 5: American Indian alone
- 6: Asian alone
- 7: Native Hawaiian or Other Pacific Islander alone
- 8: Some other race alone
- 9: Two or more races

### Sex (SEX)

The variable SEX is binary in the ACS:

- 1: Male
- 2: Female

## Models

We train diverse models on the same dataset:

### Accuracy-first model

This model uses all available features, including sensitive attributes, and is optimized purely for predictive performance (accuracy and ROC AUC). It represents a standard automated ML pipeline without fairness constraints.

### Fairness-aware model

This model excludes sensitive attributes (race and sex) from the feature set to reduce direct discrimination. While this may slightly reduce predictive accuracy, it aims to lower disparities between protected groups.

### Accuracy and fairness model (reweighting)

Rather than removing sensitive attributes, the model adopts a reweighting strategy that increases the influence of protected groups during training. This allows the model to retain access to sensitive information while mitigating disparities, resulting in more balance between accuracy and fairness.

### Adaptive reweighting model

The model applies an adaptive reweighting strategy in which the training importance of protected groups is gradually increased by different weight factors. By modifying these weights, the model gives more influence to underrepresented groups during training, allowing us to observe how fairness improves as the weights increase, and to analyze the balance between predictive accuracy and fairness.

### Fairness percentage for each group in sensitive attributes

Fairness is reported as a percentage relative to a reference group. Values below 100% indicate reduced predicted hiring opportunities compared to the reference group, while values above 100% indicate an advantage.

### Fairness-aware scoring model

Instead of selecting the model that maximizes predictive performance alone, the model ranks candidate models using a joint objective that penalizes unfairness. Concretely, the model computes a composite score AUC − λ·fairness_gap, where fairness_gap is measured using demographic parity gaps across race and sex. 

## Data download

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from folktables import ACSDataSource, ACSEmployment

# Download and load ACS data
data_source = ACSDataSource(
    survey_year='2018',
    horizon='1-Year',
    survey='person'
)

acs_data = data_source.get_data(states=["CA"], download=True)

# Define task
task = ACSEmployment

# Features, target, sensitive attributes
X, y, sensitive = task.df_to_numpy(acs_data)

feature_names = task.features
sensitive_names = task.group

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

# Pull sensitive attributes directly from ACS data (always present)
df["race"] = acs_data["RAC1P"].to_numpy()
df["sex"] = acs_data["SEX"].to_numpy()

## Train/Test split 

In [2]:
X = df[feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model 1: Prioritizes accuracy. Optimizes pure performance.

In [22]:
model_acc = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_acc.fit(X_train, y_train)

y_pred = model_acc.predict(X_test)
y_prob = model_acc.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7706386498424933
ROC AUC: 0.8446744344632616


## Fairness evaluation (post-hoc)

In [20]:
test_results = df.iloc[y_test.index].copy()
test_results["prob"] = y_prob

print("\nAverage predicted hiring probability by race:")
print(test_results.groupby("race")["prob"].mean())

print("\nAverage predicted hiring probability by sex:")
print(test_results.groupby("sex")["prob"].mean())

race_gap = (
    test_results.groupby("race")["prob"].mean().max()
    - test_results.groupby("race")["prob"].mean().min()
)

sex_gap = (
    test_results.groupby("sex")["prob"].mean().max()
    - test_results.groupby("sex")["prob"].mean().min()
)

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Average predicted probability by race:
race
1    0.452370
2    0.415184
3    0.429456
4    0.364753
5    0.416895
6    0.519927
7    0.447474
8    0.437950
9    0.365366
Name: prob, dtype: float64

Average predicted probability by sex:
sex
1    0.488672
2    0.422182
Name: prob, dtype: float64

Demographic Parity gap (race): 0.1551743155432752
Demographic Parity gap (sex): 0.06649083959274432


# Model 2: Prioritizes fairness. Eliminates sensitive attributes

In [5]:
# Reduced feature set
fair_features = [
    f for f in feature_names
    if f not in ["RAC1P", "SEX"]
]

X_fair = df[fair_features]
y = df["target"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_f = scaler.fit_transform(X_train_f)
X_test_f = scaler.transform(X_test_f)

## Fairness aware training

In [23]:
model_fair = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_fair.fit(X_train_f, y_train_f)

y_pred_f = model_fair.predict(X_test_f)
y_prob_f = model_fair.predict_proba(X_test_f)[:, 1]

print("Accuracy:", accuracy_score(y_test_f, y_pred_f))
print("ROC AUC:", roc_auc_score(y_test_f, y_prob_f))

Accuracy: 0.766766978160252
ROC AUC: 0.8374121547165523


## Fairness evaluation

In [7]:
test_results_f = df.iloc[y_test_f.index].copy()
test_results_f["prob"] = y_prob_f

print("\nAverage predicted hiring probability by race:")
print(test_results_f.groupby("race")["prob"].mean())

print("\nAverage predicted hiring probability by sex:")
print(test_results_f.groupby("sex")["prob"].mean())

race_gap_f = (
    test_results_f.groupby("race")["prob"].mean().max()
    - test_results_f.groupby("race")["prob"].mean().min()
)

sex_gap_f = (
    test_results_f.groupby("sex")["prob"].mean().max()
    - test_results_f.groupby("sex")["prob"].mean().min()
)

print("\nDemographic Parity gap (race):", race_gap_f)
print("Demographic Parity gap (sex):", sex_gap_f)


Average predicted employment probability by race:
race
1    0.454005
2    0.409847
3    0.425227
4    0.375918
5    0.413447
6    0.526811
7    0.452419
8    0.440314
9    0.364970
Name: prob, dtype: float64

Average predicted employment probability by sex:
sex
1    0.443548
2    0.469677
Name: prob, dtype: float64

Demographic Parity gap (race): 0.16184075007551096
Demographic Parity gap (sex): 0.026129337045492973


# Model 3: Prioritizes Accuracy and Fairness without eliminating the sensitive attributes

In [8]:
# Features and target
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Defining weights according to weights (reweighting)

In [9]:
train_df = df.loc[X_train.index, ["race", "sex"]]

# Initialize all weights to 1
sample_weights = np.ones(len(train_df))

# Increase weight for protected groups
sample_weights[train_df["sex"] == 2] *= 1.5
sample_weights[train_df["race"] != 1] *= 1.5

model_fair_balanced = LogisticRegression(max_iter=2000)

model_fair_balanced.fit(
    X_train_s,
    y_train,
    sample_weight=sample_weights
)

y_pred = model_fair_balanced.predict(X_test_s)
y_prob = model_fair_balanced.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Accuracy (balanced model): 0.7705770550657304
ROC AUC (balanced model): 0.8443538425722302


## Fairness Evaluation

In [10]:
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

print("\nAvg predicted hiring probability by race:")
print(test_df.groupby("race")["prob"].mean())

print("\nAvg predicted hiring probability by sex:")
print(test_df.groupby("sex")["prob"].mean())

race_gap = test_df.groupby("race")["prob"].mean().max() - test_df.groupby("race")["prob"].mean().min()
sex_gap  = test_df.groupby("sex")["prob"].mean().max()  - test_df.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_gap)
print("Demographic Parity gap (sex):", sex_gap)


Avg predicted hiring probability by race:
race
1    0.453650
2    0.413420
3    0.428759
4    0.361566
5    0.415836
6    0.521260
7    0.447367
8    0.438097
9    0.363585
Name: prob, dtype: float64

Avg predicted hiring probability by sex:
sex
1    0.490441
2    0.422103
Name: prob, dtype: float64

Demographic Parity gap (race): 0.15969390444795184
Demographic Parity gap (sex): 0.0683384153104723


# Model 4: Adaptive reweighting

In [21]:
# Prepare data
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Sensitive attributes aligned to split
train_groups = df.loc[X_train.index, ["race", "sex"]]
test_groups  = df.loc[X_test.index,  ["race", "sex"]]

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Weight multipliers to try (1.0 = no reweighting)
weight_grid = [1.0, 1.1, 1.25, 1.5, 2.0]

results = []

for w in weight_grid:
    # Base weight = 1 for everyone
    sample_weights = np.ones(len(train_groups), dtype=float)

    # Protected definitions
    is_female = (train_groups["sex"] == 2)
    is_nonwhite = (train_groups["race"] != 1)

    # Apply multiplier to protected examples (can stack -> intersection gets w*w)
    sample_weights[is_female] *= w
    sample_weights[is_nonwhite] *= w

    # Train model
    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_s, y_train, sample_weight=sample_weights)

    # Predict on test
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    # Fairness: Demographic parity gap on predicted probabilities
    test_df = test_groups.copy()
    test_df["prob"] = y_prob

    avg_by_race = test_df.groupby("race")["prob"].mean()
    avg_by_sex  = test_df.groupby("sex")["prob"].mean()

    race_dp_gap = avg_by_race.max() - avg_by_race.min()
    sex_dp_gap  = avg_by_sex.max()  - avg_by_sex.min()

    # (Optional) Fairness % for female vs male (reference male=1)
    ref_male = avg_by_sex.loc[1.0] if 1.0 in avg_by_sex.index else avg_by_sex.iloc[0]
    female_fairness_pct = (avg_by_sex.loc[2.0] / ref_male) * 100 if 2.0 in avg_by_sex.index else np.nan

    results.append({
        "weight_multiplier": w,
        "accuracy": acc,
        "roc_auc": auc,
        "race_dp_gap": race_dp_gap,
        "sex_dp_gap": sex_dp_gap,
        "female_fairness_%": female_fairness_pct
    })

# Show results 
print("\nAdaptive reweighting results:")
for row in results:
    print(
        f"w={row['weight_multiplier']}: "
        f"acc={row['accuracy']:.4f}, auc={row['roc_auc']:.4f}, "
        f"race_DP_gap={row['race_dp_gap']:.4f}, sex_DP_gap={row['sex_dp_gap']:.4f}, "
        f"female_fairness%={row['female_fairness_%']:.2f}%"
    )


Adaptive reweighting results:
w=1.0: acc=0.7705, auc=0.8447, race_DP_gap=0.1662, sex_DP_gap=0.0708, female_fairness%=85.63%
w=1.1: acc=0.7706, auc=0.8446, race_DP_gap=0.1647, sex_DP_gap=0.0702, female_fairness%=85.73%
w=1.25: acc=0.7705, auc=0.8445, race_DP_gap=0.1626, sex_DP_gap=0.0695, female_fairness%=85.87%
w=1.5: acc=0.7706, auc=0.8444, race_DP_gap=0.1597, sex_DP_gap=0.0683, female_fairness%=86.07%
w=2.0: acc=0.7699, auc=0.8439, race_DP_gap=0.1552, sex_DP_gap=0.0665, female_fairness%=86.39%


# Model 5: Fairness percentage for each sensitive attribute

## For sex attribute

In [18]:
# Build test dataframe
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

# Reference group: Male
ref_mean_sex = test_df[test_df["sex"] == 1]["prob"].mean()

print("\nFairness percentage by sex:")

for s in sorted(test_df["sex"].unique()):
    group_mean = test_df[test_df["sex"] == s]["prob"].mean()
    fairness_pct = (group_mean / ref_mean_sex) * 100
    print(f"{int(s)}: {fairness_pct:.2f}%")


Fairness percentage by sex:
1: 100.00%
2: 86.39%


## For race attribute

In [19]:
# Reference group: White
ref_mean_race = test_df[test_df["race"] == 1]["prob"].mean()

print("\nFairness percentage by race:")

for r in sorted(test_df["race"].unique()):
    group_mean = test_df[test_df["race"] == r]["prob"].mean()
    fairness_pct = (group_mean / ref_mean_race) * 100
    print(f"{int(r)}: {fairness_pct:.2f}%")


Fairness percentage by race:
1: 100.00%
2: 91.78%
3: 94.93%
4: 80.63%
5: 92.16%
6: 114.93%
7: 98.92%
8: 96.81%
9: 80.77%


# Model 6: Fairness-aware scoring

In [17]:
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]
X = df[feature_names]
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

train_groups = df.loc[X_train.index, ["race", "sex"]]
test_groups  = df.loc[X_test.index,  ["race", "sex"]]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

weight_grid = [1.0, 1.1, 1.25, 1.5, 2.0]

candidates = []

for w in weight_grid:
    sample_weights = np.ones(len(train_groups), dtype=float)

    # protected groups
    sample_weights[train_groups["sex"] == 2] *= w         # female
    sample_weights[train_groups["race"] != 1] *= w        # non-white

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train_s, y_train, sample_weight=sample_weights)

    y_prob = model.predict_proba(X_test_s)[:, 1]
    auc = roc_auc_score(y_test, y_prob)

    # fairness gaps on test
    test_df = test_groups.copy()
    test_df["prob"] = y_prob

    avg_by_race = test_df.groupby("race")["prob"].mean()
    avg_by_sex  = test_df.groupby("sex")["prob"].mean()

    race_dp_gap = float(avg_by_race.max() - avg_by_race.min())
    sex_dp_gap  = float(avg_by_sex.max()  - avg_by_sex.min())

    candidates.append({
        "w": w,
        "model": model,
        "auc": auc,
        "race_dp_gap": race_dp_gap,
        "sex_dp_gap": sex_dp_gap
    })

# 3) Fairness-aware scoring and selection

alpha = 1.0  # weight for race gap
beta  = 1.0  # weight for sex gap

lambda_grid = [0.0, 0.5, 1.0, 2.0, 5.0]  # try different trade-offs

print("\nFairness-aware scoring: score = AUC - λ*(alpha*race_gap + beta*sex_gap)\n")

best_overall = None

for lam in lambda_grid:
    best = None
    for c in candidates:
        fairness_penalty = alpha * c["race_dp_gap"] + beta * c["sex_dp_gap"]
        score = c["auc"] - lam * fairness_penalty

        if best is None or score > best["score"]:
            best = {**c, "lambda": lam, "score": score}

    print(
        f"lambda={lam}: BEST w={best['w']} | "
        f"AUC={best['auc']:.4f} | race_gap={best['race_dp_gap']:.4f} | sex_gap={best['sex_dp_gap']:.4f} | "
        f"score={best['score']:.4f}"
    )

    if best_overall is None or best["score"] > best_overall["score"]:
        best_overall = best

print("\nSelected (highest score across lambdas):")
print(
    f"lambda={best_overall['lambda']} | w={best_overall['w']} | "
    f"AUC={best_overall['auc']:.4f} | race_gap={best_overall['race_dp_gap']:.4f} | sex_gap={best_overall['sex_dp_gap']:.4f} | "
    f"score={best_overall['score']:.4f}"
)


Fairness-aware scoring: score = AUC - λ*(alpha*race_gap + beta*sex_gap)

lambda=0.0: BEST w=1.0 | AUC=0.8447 | race_gap=0.1662 | sex_gap=0.0708 | score=0.8447
lambda=0.5: BEST w=2.0 | AUC=0.8439 | race_gap=0.1552 | sex_gap=0.0665 | score=0.7331
lambda=1.0: BEST w=2.0 | AUC=0.8439 | race_gap=0.1552 | sex_gap=0.0665 | score=0.6223
lambda=2.0: BEST w=2.0 | AUC=0.8439 | race_gap=0.1552 | sex_gap=0.0665 | score=0.4006
lambda=5.0: BEST w=2.0 | AUC=0.8439 | race_gap=0.1552 | sex_gap=0.0665 | score=-0.2644

Selected (highest score across lambdas):
lambda=0.0 | w=1.0 | AUC=0.8447 | race_gap=0.1662 | sex_gap=0.0708 | score=0.8447
